In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install -q torch_geometric
# !pip install -q class_resolver
# !pip3 install pymatting

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from scipy import sparse
from scipy.sparse.linalg import eigsh
from torch.utils.data import TensorDataset, DataLoader, Subset
import random

In [ ]:
import numpy as np

feature_path = "/home/snu/Downloads/liver_dino_features.npy"

label_path = "/home/snu/Downloads/liver_dino_labels.npy"

F = np.load(
    feature_path
).astype(np.float32)

y_labels = np.load(
    label_path
).astype(np.int64)

print("Feature shape:", F.shape)

print("Label shape:", y_labels.shape)

print("\nClass distribution:")

unique, counts = np.unique(
    y_labels,
    return_counts=True
)

for cls, cnt in zip(unique, counts):

    print(f"Class {cls}: {cnt}")

Feature shape: (635, 768)
Label shape: (635,)

Class distribution:
Class 0: 200
Class 1: 435


In [ ]:
X = F
y = y_labels
np.random.seed(42)
perm = np.random.permutation(X.shape[0])
X = X[perm]
y = y[perm]
num_nodes, num_feats = X.shape
print(f"Features: {X.shape}, Labels: {y.shape}")

Features: (635, 768), Labels: (635,)


In [ ]:
def tokencut_on_features(F_array, alpha=1e-6):
    N, D = F_array.shape

    # Normalize features row-wise
    norms = np.linalg.norm(F_array, axis=1, keepdims=True) + 1e-10
    F_norm = F_array / norms

    # Cosine similarity matrix
    W = np.dot(F_norm, F_norm.T)
    W = W + alpha

    # Normalized Laplacian
    d = np.sum(W, axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-10))
    L = np.eye(N) - d_inv_sqrt @ W @ d_inv_sqrt

    L_sparse = sparse.csr_matrix(L)

    # Fiedler vector (2nd smallest eigenvector)
    vals, vecs = eigsh(L_sparse, k=2, which='SM')
    fiedler = vecs[:, 1]

    # Threshold by mean
    threshold = fiedler.mean()
    labels = (fiedler > threshold).astype(np.int64)

    return labels, fiedler

labels, scores = tokencut_on_features(F)

In [ ]:
y_pred = labels
acc = accuracy_score(y_labels, y_pred)
inv_acc = accuracy_score(y_labels, 1 - y_pred)
if inv_acc > acc:
    y_pred = 1 - y_pred
    acc = inv_acc

prec = precision_score(y_labels, y_pred)
rec = recall_score(y_labels, y_pred)
f1 = f1_score(y_labels, y_pred)

# Normalize fiedler scores for logloss
probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
logloss = log_loss(y_labels, probs)

print("===== TokenCut Results (PneumoniaMNIST) =====")
print("Accuracy Score:", acc)
print("Precision Score:", prec)
print("Recall Score:", rec)
print("F1 Score:", f1)
print("Log Loss:", logloss)

===== TokenCut Results (PneumoniaMNIST) =====
Accuracy Score: 0.6661417322834645
Precision Score: 0.8289085545722714
Recall Score: 0.6459770114942529
F1 Score: 0.7260981912144703
Log Loss: 0.6638145517583671


In [ ]:
print(y_pred)

[0 0 1 0 0 1 1 0 0 0 0 1 1 1 1 1 1 1 0 0 0 1 1 1 0 0 0 1 0 1 0 0 0 1 0 1 1
 1 0 1 0 1 0 0 0 0 0 1 1 0 1 1 1 1 0 1 1 0 0 1 0 0 1 0 0 1 1 0 1 1 0 0 1 1
 1 0 1 0 0 1 1 0 0 1 0 1 1 1 0 0 0 1 0 0 0 1 1 1 0 1 0 0 1 1 1 1 1 0 0 1 0
 0 1 1 0 1 0 1 1 0 1 0 1 1 1 1 1 0 0 1 0 1 0 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1
 1 1 1 0 0 1 0 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 0 1 1 1 0 0 1 0 1 1 0 1 0 1 0
 0 1 0 0 0 0 1 1 0 1 1 1 0 0 0 1 0 0 0 0 1 0 1 0 1 1 1 0 0 1 0 1 1 0 1 1 1
 1 0 0 0 0 0 0 1 1 1 1 0 0 0 1 1 0 1 0 0 0 1 0 1 0 1 0 1 1 0 1 0 1 1 1 1 0
 0 1 0 1 1 0 0 1 1 0 0 0 0 1 1 0 0 0 0 1 1 0 1 0 1 0 0 0 1 0 1 1 0 1 1 0 0
 1 0 1 0 1 1 0 1 0 0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 1 1 0 1 1 0 0 0 0 1 0 0 1
 0 1 1 0 1 1 0 1 0 0 1 0 0 0 1 1 1 1 1 0 1 0 0 0 0 1 0 1 0 0 1 1 0 1 1 0 0
 0 1 0 1 0 1 1 1 0 1 1 1 0 1 1 0 0 1 1 1 0 1 1 0 1 0 1 0 0 0 1 1 0 0 1 0 1
 0 1 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 0 0 1 0 0 1 1 0 0 0 1 1 1 0 1 0 0 0 1 0
 1 1 1 1 0 1 1 1 1 0 1 1 0 0 0 0 1 1 1 1 1 0 0 1 1 0 1 1 0 0 0 1 1 0 1 0 1
 1 1 1 1 0 1 0 1 1 1 0 1 

In [ ]:
print(y_labels)

[0 0 1 0 0 1 1 1 0 0 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 0 1 1 0 0 1 1 0 1 0 1 1
 1 0 1 0 1 1 1 1 1 0 1 0 0 0 1 1 1 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0 1
 0 0 1 0 0 1 1 1 1 1 0 1 1 0 0 1 0 0 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 1 1 0 0 1 0 1 1 1 0 1
 0 1 1 0 1 0 1 0 1 0 1 1 1 0 0 1 1 1 0 0 0 1 0 1 1 1 0 0 1 1 0 1 1 1 1 1 1
 0 1 0 1 0 1 1 1 0 1 1 1 1 0 1 1 0 1 0 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0
 0 1 0 1 1 0 0 0 1 1 0 0 1 0 1 1 1 1 1 1 0 1 0 1 0 1 0 1 1 1 1 1 1 1 1 1 0
 0 1 0 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1
 0 1 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 1 0 1 1 1 0 0 1 1 0 1 1 1 0 1 1 0 1 0 1
 1 1 1 0 0 0 1 1 0 1 1 0 0 1 1 1 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 0 1 1 1 1 0 0 1 0 1 1 1 0 0 1 1 0 1 1 1 0 1 0 0 1 0 1 0 0 1 1 1 1 1 1 0 1
 0 1 1 1 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0 1 1 1 1 1 1 0 0 1 1 1 0 0 0 0 1 1 1
 1 1 1 1 1 1 0 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 1 1 1 0 0 1 0 1 0 1 0 1 1 1
 1 1 1 1 1 0 0 1 1 1 0 1 

In [ ]:
num_runs = 10

acc_scores, prec_scores, rec_scores, f1_scores, log_losses = [], [], [], [], []

for run in range(num_runs):
    print(f"\n--- Run {run+1}/{num_runs} ---")
    np.random.seed(run)
    torch.manual_seed(run)

    y_pred, scores = tokencut_on_features(F)

    acc = accuracy_score(y_labels, y_pred)
    inv_acc = accuracy_score(y_labels, 1 - y_pred)
    if inv_acc > acc:
        y_pred = 1 - y_pred
        acc = inv_acc

    prec = precision_score(y_labels, y_pred, zero_division=0)
    rec = recall_score(y_labels, y_pred, zero_division=0)
    f1 = f1_score(y_labels, y_pred, zero_division=0)

    probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
    logloss = log_loss(y_labels, probs)

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)
    f1_scores.append(f1)
    log_losses.append(logloss)

    print(f"Run {run+1} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | "
          f"F1: {f1:.4f} | LogLoss: {logloss:.4f}")

print("\n================ FINAL SUMMARY ================\n")
print(f"{'Metric':>15} | {'Mean':>10} ± {'Std':<10}")
print("-" * 50)
print(f"{'Accuracy':>15} | {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print(f"{'Precision':>15} | {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"{'Recall':>15} | {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")
print(f"{'F1 Score':>15} | {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"{'Log Loss':>15} | {np.mean(log_losses):.4f} ± {np.std(log_losses):.4f}")


--- Run 1/10 ---
Run 1 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 1.0043

--- Run 2/10 ---
Run 2 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 0.6638

--- Run 3/10 ---
Run 3 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 0.6638

--- Run 4/10 ---
Run 4 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 1.0043

--- Run 5/10 ---
Run 5 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 1.0043

--- Run 6/10 ---
Run 6 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 0.6638

--- Run 7/10 ---
Run 7 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 0.6638

--- Run 8/10 ---
Run 8 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 0.6638

--- Run 9/10 ---
Run 9 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 1.0043

--- Run 10/10 ---
Run 10 | Acc: 0.6661 | Prec: 0.8289 | Rec: 0.6460 | F1: 0.7261 | LogLoss: 1.0043

================ 